# Build the results-overview deck — "IMD_Results_Overview.pptx"

New presentation structure, distinct from the internal review and GISIdeas
decks: Introduction (with a created concept graphic) → Study area and its
importance → Data used → Methodology (a pipeline diagram) → Results (each
city's predicted-map grid with a difference row against its benchmark(s), then
each city's three-technique independent-validation table) → Conclusion.

Three cells, run in order:
1. The two conceptual figures (introduction graphic, methodology pipeline).
2. The three per-city results grids (predicted + difference maps). Reads
   rasters directly and takes a few minutes.
3. The deck itself, built fresh from the Polimi template (regenerates from
   scratch every run -- do not run after manual PowerPoint edits without
   expecting to lose them).

Close the .pptx in PowerPoint (and let OneDrive finish syncing) before
running cell 3, or the save will fail with a permission error.

## 1. Methodology pipeline diagram

In [ ]:
# -*- coding: utf-8 -*-
"""Methodology pipeline diagram, v6: same spine layout and content as v5
(same-source validation dropped, only independent validation remains), but
with real polish instead of flat colour blocks -- a soft drop shadow and a
top gloss highlight on every box for depth, and a small icon badge on each
one showing what it actually is: a colour-ramp swatch for the two real
reference rasters, a satellite for Sentinel-2, a stacked-band icon for
AlphaEarth, a plot pin for EarthLabel, and a matching glyph for every step
of the pipeline. Saved to figs_deck/.
"""
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle, Polygon, FancyArrow
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import numpy as np
import contextily as cx
import rasterio
from rasterio.windows import from_bounds

# A real aerial crop (Esri World Imagery, same source as the study-area
# world map and the introduction image) for the EarthLabel icon -- shows
# an actual photo-interpreted plot instead of an abstract pin, with the
# same 3x3 sub-cell grid EarthLabel itself uses (one plot = one 10 m
# pixel, split into 9 interpreted sub-cells).
R_EARTH = 6378137.0
def _to_3857(lon, lat):
    x = R_EARTH * math.radians(lon)
    y = R_EARTH * math.log(math.tan(math.pi / 4 + math.radians(lat) / 2))
    return x, y

_plot_x, _plot_y = _to_3857(9.1897, 45.4825)
_plot_half = 35
try:
    EARTHLABEL_IMG, _ = cx.bounds2img(_plot_x - _plot_half, _plot_y - _plot_half,
                                       _plot_x + _plot_half, _plot_y + _plot_half,
                                       zoom=18, source=cx.providers.Esri.WorldImagery, ll=False)
except Exception as e:
    print(f"EarthLabel thumbnail fetch failed ({e}); icon will fall back to a plain tint.")
    EARTHLABEL_IMG = None

# A second real aerial crop, wider and centred on Milan itself, standing in
# for "what a satellite optical composite over this AOI looks like" on the
# Sentinel-2 box -- real imagery, not a drawn satellite glyph.
_s2_x, _s2_y = _to_3857(9.19, 45.4642)
_s2_half = 900
try:
    SENTINEL_IMG, _ = cx.bounds2img(_s2_x - _s2_half, _s2_y - _s2_half,
                                     _s2_x + _s2_half, _s2_y + _s2_half,
                                     zoom=15, source=cx.providers.Esri.WorldImagery, ll=False)
except Exception as e:
    print(f"Sentinel-2 thumbnail fetch failed ({e}); icon will fall back to a plain glyph.")
    SENTINEL_IMG = None

# Real Milan rasters this study actually produced/used -- CLMS and
# GHS-BUILT-S are the real training targets, and the S2-stack prediction is
# a real model output -- read straight off disk with rasterio, not redrawn.
MATEJ = r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
_RASTER_PATHS = {
    "clms": MATEJ + r"\matej_files_codes\reference_IMD\IMD_2018_CLMS_Milan.tif",
    "ghsl": r"C:\Users\user\projects\IMD-Mapping\data\GHSL_2018_Milan_UTM32N.tif",
    "pred": MATEJ + r"\IMD\outputs_S2_stack\IMD_predicted_RF_S2_Milan.tif",
}
_RASTER_CENTER = (513375, 5033780)  # AOI centre, EPSG:32632, shared by all three
_RASTER_HALF_M = 2500


def _fetch_raster_thumb(key):
    path = _RASTER_PATHS[key]
    cx0, cy0 = _RASTER_CENTER
    try:
        with rasterio.open(path) as src:
            win = from_bounds(cx0 - _RASTER_HALF_M, cy0 - _RASTER_HALF_M,
                               cx0 + _RASTER_HALF_M, cy0 + _RASTER_HALF_M,
                               transform=src.transform)
            arr = src.read(1, window=win, out_shape=(160, 160)).astype("float32")
            nodata = src.nodata
            if nodata is not None:
                arr[arr == nodata] = np.nan
            arr[(arr < 0) | (arr > 100)] = np.nan
        return arr
    except Exception as e:
        print(f"{key} raster fetch failed ({e}); icon will fall back to a synthetic ramp.")
        return None


RASTER_THUMBS = {k: _fetch_raster_thumb(k) for k in _RASTER_PATHS}
IMD_CMAP_ICON = ListedColormap(["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"])

OUT = r"C:\Users\user\projects\IMD-Mapping\figs_deck"

INK = "#1a1a1a"
BLUE = "#2E6F9E"
ACCENT = "#0B6E4F"
GHSL_C = "#C2724A"
PRED_TINT = "#EAF1F8"
TARGET_TINT = "#EAF3EC"
VAL_TINT = "#F6EFE7"
SHADOW = "#9aa0a6"

IMD_COLORS = ["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"]

plt.rcParams.update({"font.family": "DejaVu Sans"})

FIG_W, FIG_H = 17.4, 6.5
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.set_xlim(0, FIG_W)
ax.set_ylim(0, FIG_H)
ax.axis("off")


def box(x, y, w, h, text, color, tcolor="white", fs=10.5, weight="bold", zorder=3,
        icon=None):
    # soft drop shadow
    ax.add_patch(FancyBboxPatch((x + 0.045, y - 0.045), w, h,
                                  boxstyle="round,pad=0.02,rounding_size=0.12",
                                  linewidth=0, facecolor=SHADOW, alpha=0.35, zorder=zorder - 0.2))
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                        linewidth=0, facecolor=color, zorder=zorder)
    ax.add_patch(b)
    # top gloss highlight, clipped to the box shape
    gloss = FancyBboxPatch((x + w * 0.04, y + h * 0.46), w * 0.92, h * 0.48,
                            boxstyle="round,pad=0.0,rounding_size=0.10",
                            linewidth=0, facecolor="white", alpha=0.10, zorder=zorder + 0.1)
    ax.add_patch(gloss)
    # icon and text sit side by side, both centred on the box's own vertical
    # midline, so a tall two-line label never collides with the badge above it.
    if icon is not None:
        icon(x + 0.34, y + h / 2, zorder + 1)
        text_x, ha = x + 0.64, "left"
    else:
        text_x, ha = x + w / 2, "center"
    ax.text(text_x, y + h / 2, text, ha=ha, va="center", fontsize=fs, color=tcolor,
            fontweight=weight, zorder=zorder + 2, linespacing=1.25)
    return (x, y, w, h)


def _icon_badge(cx, cy, zorder, r=0.24):
    ax.add_patch(Circle((cx, cy), r, facecolor="white", edgecolor="none", alpha=0.92,
                          zorder=zorder))
    return cx, cy, r


def icon_real_raster(raster_key):
    """A real, actual-data thumbnail -- CLMS, GHS-BUILT-S and the S2-stack
    prediction are genuine rasters this study produced or used, read
    straight off disk and coloured with the same IMD ramp as every map
    later in the deck, not a synthetic gradient standing in for one."""
    def _icon(cx, cy, zorder):
        cx, cy, r = _icon_badge(cx, cy, zorder)
        arr = RASTER_THUMBS.get(raster_key)
        if arr is not None:
            ax.imshow(arr, extent=(cx - r * 0.85, cx + r * 0.85, cy - r * 0.85, cy + r * 0.85),
                      cmap=IMD_CMAP_ICON, vmin=0, vmax=100, zorder=zorder + 1,
                      clip_path=Circle((cx, cy), r * 0.85, transform=ax.transData), clip_on=True)
        else:
            cmap = LinearSegmentedColormap.from_list("imd", IMD_COLORS)
            grad = np.linspace(0, 1, 64).reshape(1, -1)
            ax.imshow(grad, extent=(cx - r * 0.72, cx + r * 0.72, cy - r * 0.5, cy + r * 0.5),
                      cmap=cmap, aspect="auto", zorder=zorder + 1,
                      clip_path=Circle((cx, cy), r * 0.85, transform=ax.transData), clip_on=True)
    return _icon


def icon_photo(cx, cy, zorder):
    """A real aerial/satellite photo of the Milan AOI -- what a satellite
    optical composite over this area actually looks like, not a drawn
    satellite glyph standing in for it."""
    cx, cy, r = _icon_badge(cx, cy, zorder)
    if SENTINEL_IMG is not None:
        ax.imshow(SENTINEL_IMG, extent=(cx - r * 0.85, cx + r * 0.85, cy - r * 0.85, cy + r * 0.85),
                  zorder=zorder + 1, aspect="auto",
                  clip_path=Circle((cx, cy), r * 0.85, transform=ax.transData), clip_on=True)
    else:
        ax.add_patch(Circle((cx, cy), r * 0.7, facecolor="#7d8794", edgecolor="none",
                              zorder=zorder + 1))


def icon_bands(cx, cy, zorder):
    """AlphaEarth is a 64-band embedding, not a viewable photo -- shown as
    what it actually is, a stack of bands, each drawn with a slight
    offset for a light stacked-cube depth."""
    cx, cy, r = _icon_badge(cx, cy, zorder)
    colors = ["#5b4c96", "#7C6FB0", "#9488c4", "#b0a6d6", "#cabfe3"]
    for i, c in enumerate(colors):
        off = (len(colors) - 1 - i) * r * 0.055
        ax.add_patch(Rectangle((cx - r * 0.6 + off, cy - r * 0.42 + off), r * 1.2, r * 0.62,
                                 facecolor=c, edgecolor="white", linewidth=0.5,
                                 zorder=zorder + 1 + i))


def icon_pin(cx, cy, zorder):
    cx, cy, r = _icon_badge(cx, cy, zorder)
    ax.add_patch(Polygon([(cx - r * 0.4, cy + r * 0.1), (cx + r * 0.4, cy + r * 0.1),
                           (cx, cy - r * 0.55)], closed=True, facecolor="#8B6A4A",
                          edgecolor="none", zorder=zorder + 1))
    ax.add_patch(Circle((cx, cy + r * 0.28), r * 0.34, facecolor="#8B6A4A",
                          edgecolor="none", zorder=zorder + 1))
    ax.add_patch(Circle((cx, cy + r * 0.28), r * 0.14, facecolor="white",
                          edgecolor="none", zorder=zorder + 2))


def icon_earthlabel(cx, cy, zorder):
    """A real photo-interpreted plot, not an abstract pin: an actual aerial
    crop with the same 3x3 sub-cell grid EarthLabel itself scores (one plot
    = one 10 m pixel = 9 interpreted sub-cells), a light classification
    tint, and a centroid marker -- this is what a plot really is."""
    half = 0.24  # matches the other icon badges' radius, for consistent sizing
    if EARTHLABEL_IMG is not None:
        ax.imshow(EARTHLABEL_IMG, extent=(cx - half, cx + half, cy - half, cy + half),
                  zorder=zorder, aspect="auto",
                  clip_path=Rectangle((cx - half, cy - half), 2 * half, 2 * half,
                                       transform=ax.transData))
    else:
        ax.add_patch(Rectangle((cx - half, cy - half), 2 * half, 2 * half,
                                 facecolor="#8B6A4A", edgecolor="none", zorder=zorder))
    ax.add_patch(Rectangle((cx - half, cy - half), 2 * half, 2 * half,
                             facecolor="#E4241E", alpha=0.16, edgecolor="none", zorder=zorder + 1))
    for i in (1, 2):
        gx = cx - half + i * (2 * half) / 3
        ax.plot([gx, gx], [cy - half, cy + half], color="white", lw=0.7, alpha=0.9,
                 zorder=zorder + 2)
        gy = cy - half + i * (2 * half) / 3
        ax.plot([cx - half, cx + half], [gy, gy], color="white", lw=0.7, alpha=0.9,
                 zorder=zorder + 2)
    ax.add_patch(Rectangle((cx - half, cy - half), 2 * half, 2 * half, fill=False,
                             edgecolor="white", linewidth=1.4, zorder=zorder + 2))
    ax.add_patch(Circle((cx, cy), half * 0.10, facecolor="#2c5fa8", edgecolor="white",
                          linewidth=0.5, zorder=zorder + 3))


def icon_scatter(cx, cy, zorder):
    """Stratified sampling, shown as it actually works: coloured strata
    (the target's own classes), sampled evenly rather than at random."""
    cx, cy, r = _icon_badge(cx, cy, zorder)
    strata = ["#1a9641", "#a6d96a", "#fdae61", "#d7191c"]
    quads = [(-1, -1), (1, -1), (-1, 1), (1, 1)]
    for (sx, sy), c in zip(quads, strata):
        ax.add_patch(Rectangle((cx + min(sx, 0) * r * 0.7, cy + min(sy, 0) * r * 0.7),
                                 r * 0.7, r * 0.7, facecolor=c, alpha=0.55, edgecolor="none",
                                 zorder=zorder + 1))
    rng = np.random.RandomState(3)
    for _ in range(10):
        dx = rng.uniform(-0.62, 0.62) * r
        dy = rng.uniform(-0.62, 0.62) * r
        ax.add_patch(Circle((cx + dx, cy + dy), r * 0.085, facecolor="white",
                              edgecolor="#2a2a2a", linewidth=0.4, zorder=zorder + 2))


def icon_extract(cx, cy, zorder):
    cx, cy, r = _icon_badge(cx, cy, zorder)
    for i in range(3):
        ax.add_patch(Rectangle((cx - r * 0.55, cy + r * 0.35 - i * r * 0.32), r * 1.1, r * 0.18,
                                 facecolor="#8a8f96", edgecolor="none", zorder=zorder + 1))
    ax.annotate("", xy=(cx, cy - r * 0.65), xytext=(cx, cy - r * 0.05),
                arrowprops=dict(arrowstyle="-|>", color="#4a4a4a", lw=1.3), zorder=zorder + 1)


def icon_split(cx, cy, zorder):
    """The real 1 km spatial-block split, not an arbitrary half-and-half
    rectangle: a block checkerboard, alternating train and test."""
    cx, cy, r = _icon_badge(cx, cy, zorder)
    n = 3
    cell = r * 1.3 / n
    x0, y0 = cx - r * 0.65, cy - r * 0.65
    train, test = "#4a4a4a", "#e08a3c"
    pattern = [[1, 0, 1], [0, 1, 0], [1, 0, 1]]
    for i in range(n):
        for j in range(n):
            c = train if pattern[i][j] else test
            ax.add_patch(Rectangle((x0 + j * cell, y0 + i * cell), cell * 0.92, cell * 0.92,
                                     facecolor=c, edgecolor="none", zorder=zorder + 1))


def icon_tree(cx, cy, zorder):
    """An ensemble of trees, not one -- Random Forest is a forest."""
    cx, cy, r = _icon_badge(cx, cy, zorder)
    for tx, scale, alpha in [(-r * 0.34, 0.62, 0.75), (r * 0.34, 0.62, 0.75), (0, 0.85, 1.0)]:
        base = cy - r * 0.55
        ax.plot([cx + tx, cx + tx], [base, base + r * 0.42 * scale], color="#6b4a30",
                 lw=1.3 * scale, solid_capstyle="round", zorder=zorder + 1, alpha=alpha)
        ax.add_patch(Circle((cx + tx, base + r * 0.62 * scale), r * 0.30 * scale, facecolor=BLUE,
                              edgecolor="none", zorder=zorder + 1, alpha=alpha))


def icon_map(cx, cy, zorder):
    icon_real_raster("pred")(cx, cy, zorder)


def icon_check(cx, cy, zorder):
    cx, cy, r = _icon_badge(cx, cy, zorder)
    ax.add_patch(Circle((cx, cy), r * 0.78, facecolor="none", edgecolor="#8B6A4A",
                          linewidth=1.6, zorder=zorder + 1))
    ax.plot([cx - r * 0.32, cx - r * 0.06, cx + r * 0.38], [cy - r * 0.02, cy - r * 0.28, cy + r * 0.30],
             color="#8B6A4A", lw=1.8, solid_capstyle="round", solid_joinstyle="round", zorder=zorder + 1)


def group_card(x, y, w, h, tint, label, dashed=False):
    ls = "dashed" if dashed else "solid"
    c = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.10",
                        linewidth=1.3, linestyle=ls, edgecolor="#b7b7b7",
                        facecolor=tint, zorder=0)
    ax.add_patch(c)
    ax.text(x + w / 2, y + h - 0.14, label, ha="center", va="top", fontsize=11,
            color=INK, fontweight="bold", zorder=1)


def varrow(b1, b2, color="#555555", lw=2.0, ls="solid", zorder=1):
    x1 = b1[0] + b1[2] / 2
    y1 = b1[1]
    x2 = b2[0] + b2[2] / 2
    y2 = b2[1] + b2[3]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                shrinkA=2, shrinkB=2, linestyle=ls), zorder=zorder)


def harrow(b1, b2, color="#555555", lw=2.0, zorder=1):
    x1 = b1[0] + b1[2]
    y1 = b1[1] + b1[3] / 2
    x2 = b2[0]
    y2 = b2[1] + b2[3] / 2
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                shrinkA=2, shrinkB=2), zorder=zorder)


# ============================================================= spine columns
GAP = 0.25
col1_w, col2_w, col3_w, col4_w, col5_w = 4.20, 4.05, 2.60, 2.60, 2.60
col1_x = 0.15
col2_x = col1_x + col1_w + GAP
col3_x = col2_x + col2_w + GAP
col4_x = col3_x + col3_w + GAP
col5_x = col4_x + col4_w + GAP

# --------------------------------------------------------------- Row 1: data
y1, h1 = 4.90, 1.05
card_y, card_h = y1 - 0.08, h1 + 0.45

group_card(col1_x - 0.05, card_y, col1_w + 0.10, card_h, TARGET_TINT, "TRAINING TARGETS")
group_card(col2_x - 0.05, card_y, col2_w + 0.10, card_h, PRED_TINT, "PREDICTORS")
group_card(col5_x - 0.05, card_y, col5_w + 0.10, card_h, VAL_TINT, "VALIDATION ONLY", dashed=True)

d3 = box(col1_x + 0.10, y1, 1.85, h1, "CLMS 2018\n(Milan)", ACCENT, fs=9.3, icon=icon_real_raster("clms"))
d4 = box(col1_x + 2.10, y1, 2.00, h1, "GHS-BUILT-S 2018\n(Milan, Hanoi, HCMC)", GHSL_C, fs=7.5,
         icon=icon_real_raster("ghsl"))
d1 = box(col2_x + 0.10, y1, 1.80, h1, "Sentinel-2\ncomposites, 2018", BLUE, fs=7.9,
         icon=icon_photo)
d2 = box(col2_x + 2.05, y1, 1.90, h1, "AlphaEarth\nembeddings, 2018", "#7C6FB0", fs=8.4,
         icon=icon_bands)
d5 = box(col5_x + 0.10, y1, 2.40, h1, "EarthLabel\nplots, 2018", "#8B6A4A", fs=9.6,
         icon=icon_earthlabel)

# ------------------------------------------------------------- Row 2: spine
y2, h2 = 3.10, 1.05
s_sample = box(col1_x, y2, col1_w, h2, "Stratified spatial\nsample", "#4a4a4a", fs=10.4,
               icon=icon_scatter)
s_extract = box(col2_x, y2, col2_w, h2, "Extract predictor\nvalues", "#4a4a4a", fs=10.4,
                icon=icon_extract)
s_split = box(col3_x, y2, col3_w, h2, "Train / test\nsplit", "#4a4a4a", fs=10.4, icon=icon_split)
s_train = box(col4_x, y2, col4_w, h2, "Random\nforest", BLUE, fs=10.4, icon=icon_tree)
s_predict = box(col5_x, y2, col5_w, h2, "Raster\nprediction", BLUE, fs=10.4, icon=icon_map)

varrow(d3, s_sample); varrow(d4, s_sample)
varrow(d1, s_extract); varrow(d2, s_extract)
harrow(s_sample, s_extract)
harrow(s_extract, s_split)
harrow(s_split, s_train)
harrow(s_train, s_predict)

# --------------------------------------------------------- Row 3: validation
y3, h3 = 0.65, 1.25
v2 = box(col5_x, y3, col5_w, h3, "Independent\nvalidation", "#8B6A4A", fs=11.2, icon=icon_check)

# EarthLabel feeds independent validation, passing behind the Raster
# prediction box it shares a column with (dashed, low zorder). Offset from
# centre so it lands beside, not on top of, the prediction arrow below.
ex, ey = d5[0] + d5[2] / 2, d5[1]
vx, vy = v2[0] + v2[2] - 0.55, v2[1] + v2[3]
ax.annotate("", xy=(vx, vy), xytext=(ex, ey),
            arrowprops=dict(arrowstyle="-|>", color="#8B6A4A", lw=1.9,
                            shrinkA=2, shrinkB=2, linestyle=(0, (5, 3))), zorder=0.5)

# Raster prediction feeds independent validation directly below it.
varrow(s_predict, v2)

fig.tight_layout()
fig.savefig(f"{OUT}/methodology_pipeline.png", dpi=200, bbox_inches="tight",
            facecolor="white")
plt.close(fig)
print("saved methodology_pipeline.png")


## 1b. Merged introduction figure (what impervious surface is, and why it is measured)

In [ ]:
# -*- coding: utf-8 -*-
"""Introduction graphic, v9: real aerial photography instead of drawn
vector scenery -- matplotlib cannot render photorealistically, so this
uses actual high-resolution satellite/aerial imagery (Esri World Imagery,
the same source as the study-area world map) of two real places in Milan,
one of the three study cities: a park (near Parco Sempione) for the
pervious reference, and a dense red-roofed residential block for the
impervious reference. The same technical annotations from the previous
version -- thermometer, process-circle icon, PERVIOUS/IMPERVIOUS badge,
imperviousness badge, hydrology icon -- sit on top, with a light colour
wash tying the two photos to the same green/red language as the rest of
the figure. Saved to figs_deck/imd_concept_full.png.
"""
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
import contextily as cx
import numpy as np

OUT = r"C:\Users\user\projects\IMD-Mapping\figs_deck"

INK = "#1c1c1c"
MUTED = "#5a5a5a"
CARD = "#ffffff"
SHADOW = "#c9cdd2"
BLUE = "#2c7bc9"
BLUE_DK = "#194b7a"
RED = "#e0453a"
GREEN_BADGE = "#1a7a42"
RED_BADGE = "#b0301f"

IMD_COLORS = ["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"]
IMD_GREEN, IMD_RED = IMD_COLORS[0], IMD_COLORS[-1]

# Real places in Milan (one of the three study cities), not staged renders.
PARK_LON, PARK_LAT, PARK_HALF_M = 9.1755, 45.4750, 260
URBAN_LON, URBAN_LAT, URBAN_HALF_M = 9.1885, 45.4820, 260

plt.rcParams["font.family"] = "DejaVu Sans"

R_EARTH = 6378137.0
def to_3857(lon, lat):
    x = R_EARTH * math.radians(lon)
    y = R_EARTH * math.log(math.tan(math.pi / 4 + math.radians(lat) / 2))
    return x, y


def hgrad(ax, x0, x1, y0, y1, colors, zorder=0):
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    cmap = LinearSegmentedColormap.from_list("h", colors)
    ax.imshow(grad, extent=(x0, x1, y0, y1), origin="lower", aspect="auto",
              cmap=cmap, zorder=zorder)


def thermometer(ax, cx_, y0, h, fill_frac, color, label, zorder=8):
    w = 0.55
    ax.add_patch(FancyBboxPatch((cx_ - w / 2, y0), w, h, boxstyle="round,pad=0.02,rounding_size=0.25",
                                  facecolor="white", edgecolor="#8a8f96", linewidth=1.1, zorder=zorder))
    fh = h * fill_frac
    ax.add_patch(FancyBboxPatch((cx_ - w / 2 + 0.08, y0 + 0.08), w - 0.16, max(fh - 0.08, 0.05),
                                  boxstyle="round,pad=0.0,rounding_size=0.12",
                                  facecolor=color, edgecolor="none", zorder=zorder + 1))
    ax.add_patch(Circle((cx_, y0 - 0.32), 0.34, facecolor="white", edgecolor="#8a8f96",
                          linewidth=1.1, zorder=zorder))
    ax.add_patch(Circle((cx_, y0 - 0.32), 0.20, facecolor=color, edgecolor="none", zorder=zorder + 1))
    ax.text(cx_, y0 - 0.85, label, ha="center", va="top", fontsize=8.6, color="white",
             fontweight="bold", zorder=zorder + 1, linespacing=1.2,
             path_effects=[pe.withStroke(linewidth=2.6, foreground="#00000090")])


def process_circle(ax, cx_, cy, r, ring_color, icon="tree"):
    ax.add_patch(Circle((cx_, cy), r + 0.12, facecolor="white", edgecolor=ring_color,
                          linewidth=2.4, zorder=8))
    if icon == "tree":
        ax.add_patch(Circle((cx_, cy + r * 0.28), r * 0.55, color="#5c8a4a", zorder=9))
        ax.plot([cx_, cx_], [cy - r * 0.5, cy + r * 0.05], color="#6b4a30", lw=2.4, zorder=9,
                 solid_capstyle="round")
        for dx, y1 in [(-r * 0.32, cy - r * 0.9), (0, cy - r * 1.0), (r * 0.32, cy - r * 0.9)]:
            ax.annotate("", xy=(cx_ + dx, y1), xytext=(cx_ + dx, cy - r * 0.4),
                        arrowprops=dict(arrowstyle="-|>", color=BLUE_DK, lw=1.3), zorder=10)
    else:
        ax.add_patch(Rectangle((cx_ - r * 0.55, cy - r * 0.55), r * 1.1, r * 1.1,
                                 color="#7d8794", ec="#4b525c", lw=1.0, zorder=9))
        ax.add_patch(Rectangle((cx_ - r * 0.85, cy - r * 0.95), r * 1.7, r * 0.22,
                                 color="#5c6673", zorder=9))
        for dx, y1 in [(-r * 0.32, cy - r * 1.15), (0, cy - r * 1.25), (r * 0.32, cy - r * 1.15)]:
            ax.annotate("", xy=(cx_ + dx, cy - r * 0.75), xytext=(cx_ + dx, y1),
                        arrowprops=dict(arrowstyle="-", color=BLUE_DK, lw=1.3), zorder=10)
            ax.plot([cx_ + dx], [cy - r * 0.85], marker="o", ms=3, color=BLUE, zorder=10)


def badge_pct(ax, x, y, label, sub, bg, fg="white"):
    ax.add_patch(FancyBboxPatch((x, y), 2.2, 0.85, boxstyle="round,pad=0.02,rounding_size=0.10",
                                  facecolor=bg, edgecolor="none", zorder=8))
    ax.text(x + 1.1, y + 0.56, label, ha="center", va="center", fontsize=13.5,
             fontweight="bold", color=fg, zorder=9)
    ax.text(x + 1.1, y + 0.20, sub, ha="center", va="center", fontsize=7.6, color=fg,
             zorder=9, linespacing=1.2)


def arrow_label(ax, x, y, text, color):
    ax.text(x, y, text, ha="center", fontsize=9.3, fontweight="bold", color=color, zorder=7,
             path_effects=[pe.withStroke(linewidth=3, foreground="white")])


fig = plt.figure(figsize=(15.6, 7.9))
fig.patch.set_facecolor("white")

fig.text(0.5, 0.975,
          "$\\mathbf{Impervious\\ surface}$: ground sealed enough that rain cannot soak "
          "through it \u2014 roofs, roads, pavements, parking lots.",
          ha="center", va="center", fontsize=13.7, color=INK)

CARD_X, CARD_Y, CARD_W, CARD_H = 0.03, 0.235, 0.94, 0.685
fig.add_artist(FancyBboxPatch((CARD_X + 0.010, CARD_Y - 0.010), CARD_W, CARD_H,
                boxstyle="round,pad=0.0,rounding_size=0.010", transform=fig.transFigure,
                facecolor=SHADOW, edgecolor="none", zorder=1))
fig.add_artist(FancyBboxPatch((CARD_X, CARD_Y), CARD_W, CARD_H,
                boxstyle="round,pad=0.0,rounding_size=0.010", transform=fig.transFigure,
                facecolor=CARD, edgecolor="#e2e5e9", linewidth=1.1, zorder=2))

PANEL_W = (CARD_W - 0.03) / 2
gap = 0.03
xL = CARD_X + 0.008
xR = xL + PANEL_W + gap

strip_y = CARD_Y + 0.052
panel_y0 = strip_y + 0.032 + 0.028
panel_h = (CARD_Y + CARD_H - 0.06) - panel_y0

fig.text(xL + PANEL_W / 2, CARD_Y + CARD_H - 0.045, "RURAL REFERENCE  \u00b7  LCZ D / G",
          ha="center", fontsize=13, fontweight="bold", color=INK)
fig.text(xR + PANEL_W / 2, CARD_Y + CARD_H - 0.045, "COMPACT URBAN  \u00b7  LCZ 2 / 3",
          ha="center", fontsize=13, fontweight="bold", color=INK)

axL = fig.add_axes([xL, panel_y0, PANEL_W, panel_h])
axR = fig.add_axes([xR, panel_y0, PANEL_W, panel_h])
# both default to a zorder below the card's white background (zorder=2)
# and would render invisibly behind it otherwise.
axL.set_zorder(3)
axR.set_zorder(3)

# ---- real aerial imagery, both from Milan (one of the three study cities) --
pxL, pyL = to_3857(PARK_LON, PARK_LAT)
axL.set_xlim(pxL - PARK_HALF_M, pxL + PARK_HALF_M)
axL.set_ylim(pyL - PARK_HALF_M, pyL + PARK_HALF_M)
cx.add_basemap(axL, source=cx.providers.Esri.WorldImagery, attribution=False, zoom=18)

pxR, pyR = to_3857(URBAN_LON, URBAN_LAT)
axR.set_xlim(pxR - URBAN_HALF_M, pxR + URBAN_HALF_M)
axR.set_ylim(pyR - URBAN_HALF_M, pyR + URBAN_HALF_M)
cx.add_basemap(axR, source=cx.providers.Esri.WorldImagery, attribution=False, zoom=18)

for a in (axL, axR):
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values():
        s.set_visible(False)

# a light colour wash over each photo, ties both back to the green/red
# pervious-impervious language used everywhere else in this deck
axL.add_patch(Rectangle((pxL - PARK_HALF_M, pyL - PARK_HALF_M), PARK_HALF_M * 2,
                          PARK_HALF_M * 2, facecolor="#2f8f52", alpha=0.14, zorder=5))
axR.add_patch(Rectangle((pxR - URBAN_HALF_M, pyR - URBAN_HALF_M), URBAN_HALF_M * 2,
                          URBAN_HALF_M * 2, facecolor="#c94a2e", alpha=0.16, zorder=5))

# A second, transparent axes stacked on top of each photo, in a plain 0-10
# frame, carries the annotation layer -- keeps that layout code identical
# to the drawn-scenery version instead of fighting the photo's own CRS.
annL = fig.add_axes([xL, panel_y0, PANEL_W, panel_h])
annR = fig.add_axes([xR, panel_y0, PANEL_W, panel_h])
for a in (annL, annR):
    a.set_zorder(6); a.patch.set_alpha(0)
    a.set_xlim(0, 10); a.set_ylim(0, 10); a.axis("off")

# ================= RURAL / PERVIOUS annotations ============================
thermometer(annL, 1.1, 7.6, 1.7, 0.22, RED, "Cooler\nsurface")
for ex in (3.6, 5.0, 6.4):
    annL.annotate("", xy=(ex, 9.3), xytext=(ex, 7.7),
                  arrowprops=dict(arrowstyle="-|>", color=BLUE_DK, lw=2.0, alpha=0.9), zorder=6)
arrow_label(annL, 5, 9.55, "EVAPOTRANSPIRATION", BLUE_DK)

process_circle(annL, 8.3, 6.6, 0.85, GREEN_BADGE, icon="tree")

annL.add_patch(FancyBboxPatch((0.35, 3.55), 4.1, 0.95, boxstyle="round,pad=0.03,rounding_size=0.08",
                                facecolor=GREEN_BADGE, edgecolor="none", zorder=8, alpha=0.94))
annL.text(2.4, 4.28, "PERVIOUS", ha="center", fontsize=10, fontweight="bold", color="white", zorder=9)
annL.text(2.4, 3.85, "Enables water & energy transfer", ha="center", fontsize=7.6,
           color="white", zorder=9)

badge_pct(annL, 0.35, 0.35, "10%", "LOW IMPERVIOUSNESS", IMD_GREEN)

annL.add_patch(Circle((8.5, 1.15), 0.62, facecolor="white", edgecolor="#8a8f96", lw=1.0, zorder=8))
annL.add_patch(Rectangle((8.15, 1.15), 0.7, 0.35, color="#b98e5e", zorder=9))
annL.annotate("", xy=(8.5, 0.85), xytext=(8.5, 1.5),
              arrowprops=dict(arrowstyle="-|>", color=BLUE_DK, lw=1.4), zorder=10)
annL.text(8.5, 0.35, "Natural hydrology", ha="center", fontsize=7.6, color="white",
           fontweight="bold", zorder=9, path_effects=[pe.withStroke(linewidth=2.6, foreground="#00000090")])

# ================= URBAN / IMPERVIOUS annotations ===========================
thermometer(annR, 8.9, 7.6, 1.7, 0.85, RED, "Hotter\nsurface")
for dx in (3.4, 4.6):
    annR.annotate("", xy=(dx, 7.5), xytext=(dx, 8.9),
                  arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2.0, alpha=0.9), zorder=6)
for dx in (5.6, 6.8):
    annR.annotate("", xy=(dx, 9.3), xytext=(dx, 7.6),
                  arrowprops=dict(arrowstyle="-|>", color=RED, lw=2.2, alpha=0.95), zorder=6)
arrow_label(annR, 3.6, 9.55, "RUNOFF", BLUE_DK)
arrow_label(annR, 6.6, 9.55, "RE-RADIATION", RED)

process_circle(annR, 1.5, 6.6, 0.85, RED_BADGE, icon="building")

annR.add_patch(FancyBboxPatch((5.35, 3.55), 4.3, 0.95, boxstyle="round,pad=0.03,rounding_size=0.08",
                                facecolor=RED_BADGE, edgecolor="none", zorder=8, alpha=0.94))
annR.text(7.5, 4.28, "IMPERVIOUS", ha="center", fontsize=10, fontweight="bold", color="white", zorder=9)
annR.text(7.5, 3.85, "Blocks transfer, maximises heat", ha="center", fontsize=7.6,
           color="white", zorder=9)

badge_pct(annR, 7.4, 0.35, "90%", "HIGH IMPERVIOUSNESS", IMD_RED)

annR.add_patch(Circle((1.5, 1.15), 0.62, facecolor="white", edgecolor="#8a8f96", lw=1.0, zorder=8))
annR.add_patch(Rectangle((1.15, 0.98), 0.7, 0.32, color="#6d7178", zorder=9))
annR.plot([1.15, 1.85], [1.30, 1.30], color="#3c424b", lw=1.6, zorder=10)
annR.text(1.5, 0.35, "Sealed hydrology", ha="center", fontsize=7.6, color="white",
           fontweight="bold", zorder=9, path_effects=[pe.withStroke(linewidth=2.6, foreground="#00000090")])

# ---- shared IMD colour-ramp strip -----------------------------------------
axs = fig.add_axes([CARD_X + 0.05, strip_y, CARD_W - 0.10, 0.032])
axs.set_zorder(5); axs.patch.set_alpha(0)
axs.set_xlim(0, 30); axs.axis("off")
hgrad(axs, 0, 30, 0, 1, IMD_COLORS, zorder=1)
axs.add_patch(Rectangle((0, 0), 30, 1, fill=False, edgecolor="white", lw=1.0, zorder=2))
fig.text(CARD_X + 0.055, strip_y - 0.028, "0% impervious", ha="left", fontsize=9, color=MUTED)
fig.text(CARD_X + CARD_W - 0.055, strip_y - 0.028, "100% impervious", ha="right", fontsize=9,
          color=MUTED)
fig.text(CARD_X + CARD_W / 2, strip_y - 0.028, "the scale this study maps, city-wide",
          ha="center", fontsize=9, color=MUTED, style="italic")

# ---- imagery credit (this is real Milan aerial imagery, not a render) -----
fig.text(CARD_X + CARD_W - 0.008, CARD_Y + CARD_H - 0.008,
          "Aerial imagery: Esri World Imagery \u00b7 Milan", ha="right", va="top",
          fontsize=7, color="#9a9fa5", style="italic")

# The former on-image caption bullets now live as real, editable text
# on the slide itself (see the deck-assembly cell) instead of being
# baked into this PNG.
fig.savefig(f"{OUT}/imd_concept_full.png", dpi=210, bbox_inches="tight",
            facecolor="white")
plt.close(fig)
print("saved imd_concept_full.png")


## 1c. World map figure (Study area section)

In [ ]:
# -*- coding: utf-8 -*-
"""World map, v7: lat/long as real axis ticks outside each map frame (dark
text on white, standard cartographic convention) instead of light text
overlaid on the satellite imagery -- the overlay approach kept losing
contrast against busy imagery and colliding with neighbouring panels'
labels. A faint dotted grid stays on the imagery itself as a reference.
Builds on v6 otherwise: each inset states its reference product and real
AOI size; the main map states the Milan-Hanoi distance and a colour-key
legend; Milan's inset on the left, Hanoi/HCMC stacked on the right;
satellite basemap; bright country outlines; red AOI boxes.
Saved to figs_deck/.
"""
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects
from matplotlib.patches import Rectangle, ConnectionPatch
import geopandas as gpd
from shapely.geometry import box
import contextily as cx

REPO = r"C:\Users\user\projects\IMD-Mapping"
OUT = f"{REPO}\\figs_deck\\study_area_world_map.png"
GEOJSON = f"{REPO}\\data\\naturalearth\\ne_110m_admin_0_countries.geojson"

INK = "#1a1a1a"
ITALY_C = "#FFD500"
VIETNAM_C = "#00E5FF"
AOI_C = "#E4241E"
TICK_C = "#3a3a3a"

# name, country, lon, lat, colour, EPSG, AOI (left, bottom, right, top), reference product
CITIES = [
    ("Milan", "Italy", 9.19, 45.4642, ITALY_C, "EPSG:32632",
     (474450.0, 5004840.0, 552300.0, 5062720.0), "CLMS reference"),
    ("Hanoi", "Vietnam", 105.8542, 21.0285, VIETNAM_C, "EPSG:32648",
     (573320.0, 2310700.0, 603330.0, 2340710.0), "GHS-BUILT-S reference"),
    ("Ho Chi Minh City", "Vietnam", 106.6297, 10.8231, VIETNAM_C, "EPSG:32648",
     (670870.0, 1177160.0, 700880.0, 1207170.0), "GHS-BUILT-S reference"),
]

world = gpd.read_file(GEOJSON)
name_col = "NAME" if "NAME" in world.columns else "ADMIN"
italy = world[world[name_col] == "Italy"].to_crs(epsg=3857)
vietnam = world[world[name_col] == "Vietnam"].to_crs(epsg=3857)

pins = gpd.GeoDataFrame(
    {"city": [c[0] for c in CITIES]},
    geometry=gpd.points_from_xy([c[2] for c in CITIES], [c[3] for c in CITIES]),
    crs="EPSG:4326",
).to_crs(epsg=3857)
pins_ll = {c[0]: (c[2], c[3]) for c in CITIES}

# Spherical Web Mercator forward/inverse projection (matches EPSG:3857).
R = 6378137.0
def lon_to_x(lon):
    return R * math.radians(lon)
def lat_to_y(lat):
    lat = max(min(lat, 85.05), -85.05)
    return R * math.log(math.tan(math.pi / 4 + math.radians(lat) / 2))
def x_to_lon(x):
    return math.degrees(x / R)
def y_to_lat(y):
    return math.degrees(2 * math.atan(math.exp(y / R)) - math.pi / 2)


def lon_label(lon):
    return f"{abs(lon):g}\u00b0{'E' if lon > 0 else ('W' if lon < 0 else '')}"


def lat_label(lat):
    return f"{abs(lat):g}\u00b0{'N' if lat > 0 else ('S' if lat < 0 else '')}"


def set_ticks(ax, lons, lats, fontsize, lon_top=False, grid_color="white"):
    """Real matplotlib ticks -- dark text outside the axes frame on white,
    not text overlaid on the satellite imagery -- plus a faint dotted
    reference grid drawn on the imagery itself."""
    ax.set_xticks([lon_to_x(v) for v in lons])
    ax.set_xticklabels([lon_label(v) for v in lons], fontsize=fontsize, color=TICK_C,
                        fontweight="bold")
    ax.set_yticks([lat_to_y(v) for v in lats])
    ax.set_yticklabels([lat_label(v) for v in lats], fontsize=fontsize, color=TICK_C,
                        fontweight="bold")
    if lon_top:
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
    ax.tick_params(axis="both", direction="out", length=4, width=1.0,
                    color="#888888", pad=5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#888888")
        spine.set_linewidth(0.9)
    ax.grid(True, color=grid_color, alpha=0.6, linestyle=(0, (1, 3)), linewidth=0.8, zorder=2.5)


def nice_step(span_deg, target_lines=3.5):
    for candidate in (0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 2.0, 5.0):
        if span_deg / candidate <= target_lines:
            return candidate
    return 10.0


def graticule_values(lo, hi, step):
    lo_r = math.ceil(lo / step) * step
    hi_r = math.floor(hi / step) * step
    if hi_r < lo_r:
        return [round((lo + hi) / 2 / step) * step]
    n = int(round((hi_r - lo_r) / step))
    return [round(lo_r + i * step, 3) for i in range(n + 1)]


def haversine_km(lon1, lat1, lon2, lat2):
    R_km = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R_km * math.asin(math.sqrt(a))

milan_hanoi_km = haversine_km(9.19, 45.4642, 105.8542, 21.0285)

fig = plt.figure(figsize=(21.0, 9.6))
fig.patch.set_facecolor("white")

# ---- main world map, centred, satellite basemap ---------------------------
ax = fig.add_axes([0.300, 0.075, 0.420, 0.83])

lon_lo, lon_hi = -30, 145
lat_lo, lat_hi = -38, 62
extent = gpd.GeoDataFrame(geometry=[box(lon_lo, lat_lo, lon_hi, lat_hi)], crs="EPSG:4326").to_crs(epsg=3857)
xb = extent.total_bounds
ax.set_xlim(xb[0], xb[2])
ax.set_ylim(xb[1], xb[3])

ax.set_facecolor("#f4f4f4")
world_3857 = world.to_crs(epsg=3857)
world_3857.plot(ax=ax, color="#c9c9c9", edgecolor="#c9c9c9", linewidth=0.3, zorder=1)

italy.boundary.plot(ax=ax, color=ITALY_C, linewidth=2.2, zorder=3)
vietnam.boundary.plot(ax=ax, color=VIETNAM_C, linewidth=2.2, zorder=3)

set_ticks(ax, [-30, 0, 30, 60, 90, 120], [-30, 0, 30, 60], fontsize=13.5, lon_top=True,
          grid_color="#a8a8a8")

for (city, country, lon, lat, color, *_rest), pt in zip(CITIES, pins.geometry):
    ax.scatter([pt.x], [pt.y], s=90, color=color, edgecolor="white", linewidth=1.8, zorder=5)
    ax.scatter([pt.x], [pt.y], s=320, facecolor="none", edgecolor=color, linewidth=1.6, zorder=5)

# ---- inset panels: Milan on the LEFT, Hanoi + HCMC stacked on the RIGHT ---
inset_rects = {
    "Milan": [0.045, 0.255, 0.185, 0.46],
    "Hanoi": [0.775, 0.505, 0.205, 0.40],
    "Ho Chi Minh City": [0.775, 0.045, 0.205, 0.40],
}

for city, country, lon, lat, color, epsg, (l, b, r, t), ref_label in CITIES:
    axins = fig.add_axes(inset_rects[city])

    aoi = gpd.GeoDataFrame(geometry=[box(l, b, r, t)], crs=epsg).to_crs(epsg=3857)
    bounds = aoi.total_bounds
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]
    pad_x, pad_y = w * 0.5, h * 0.5
    axins.set_xlim(bounds[0] - pad_x, bounds[2] + pad_x)
    axins.set_ylim(bounds[1] - pad_y, bounds[3] + pad_y)

    try:
        cx.add_basemap(axins, source=cx.providers.Esri.WorldStreetMap,
                        attribution=False, zoom="auto")
    except Exception as e:
        print(f"{city}: basemap fetch failed ({e}); leaving inset blank.")

    lon0, lon1 = x_to_lon(axins.get_xlim()[0]), x_to_lon(axins.get_xlim()[1])
    lat0, lat1 = y_to_lat(axins.get_ylim()[0]), y_to_lat(axins.get_ylim()[1])
    step = nice_step(max(lon1 - lon0, lat1 - lat0))
    set_ticks(axins, graticule_values(lon0, lon1, step), graticule_values(lat0, lat1, step),
              fontsize=9.5)

    aoi.boundary.plot(ax=axins, color=AOI_C, linewidth=2.8, zorder=5)
    for spine in axins.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3.0)
    axins.set_title(f"{city}, {country}", fontsize=12, fontweight="bold",
                     color=INK, pad=8)

    aoi_w_km = (r - l) / 1000.0
    aoi_h_km = (t - b) / 1000.0
    axins.text(0.98, 0.97, f"AOI \u00b7 {aoi_w_km:.0f} \u00d7 {aoi_h_km:.0f} km",
                transform=axins.transAxes, fontsize=9,
                fontweight="bold", color="white", va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.25", facecolor=AOI_C, edgecolor="none"),
                zorder=6)

    side_x = 0.0 if inset_rects[city][0] > 0.5 else 1.0
    lon_p, lat_p = pins_ll[city]
    pin_xy = gpd.GeoDataFrame(geometry=[gpd.points_from_xy([lon_p], [lat_p])[0]],
                               crs="EPSG:4326").to_crs(epsg=3857).geometry[0]
    con = ConnectionPatch(xyA=(pin_xy.x, pin_xy.y), coordsA=ax.transData,
                           xyB=(side_x, 0.5), coordsB=axins.transAxes,
                           color=color, lw=1.6, linestyle=(0, (4, 2)), zorder=3)
    fig.add_artist(con)

fig.savefig(OUT, dpi=190, facecolor="white", bbox_inches="tight")
plt.close(fig)
print("saved", OUT)


## 1d. AOI street-level figure (Study area section)

In [ ]:
# -*- coding: utf-8 -*-
"""AOI street-view figure, v2: matches the reference layout the user shared
-- the AOI box sits inside a wider visible street map (not edge to edge),
with a proper map-style legend swatch in the corner naming the boundary,
rather than a coloured frame around the whole panel. Three panels, Milan /
Hanoi / HCMC, same colour coding as the world map. Saved to figs_deck/.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import geopandas as gpd
from shapely.geometry import box
import contextily as cx

OUT = r"C:\Users\user\projects\IMD-Mapping\figs_deck\study_area_aoi_streetview.png"

INK = "#1a1a1a"
ITALY_C = "#0B6E4F"
VIETNAM_C = "#C2724A"

# (city, country, EPSG, left, bottom, right, top) -- exact raster/AOI extents
CITIES = [
    ("Milan", "Italy, 2018", "EPSG:32632", 474450.0, 5004840.0, 552300.0, 5062720.0, ITALY_C),
    ("Hanoi", "Vietnam, 2018", "EPSG:32648", 573320.0, 2310700.0, 603330.0, 2340710.0, VIETNAM_C),
    ("Ho Chi Minh City", "Vietnam, 2018", "EPSG:32648", 670870.0, 1177160.0, 700880.0, 1207170.0, VIETNAM_C),
]

fig, axes = plt.subplots(1, 3, figsize=(16.8, 6.4))

for ax, (city, sub, epsg, l, b, r, t, color) in zip(axes, CITIES):
    aoi = gpd.GeoDataFrame(geometry=[box(l, b, r, t)], crs=epsg).to_crs(epsg=3857)
    bounds = aoi.total_bounds
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]
    # Wide margin so the boundary reads as a highlighted sub-area within a
    # larger visible map, not a box that fills the whole panel.
    pad_x, pad_y = w * 0.55, h * 0.55
    ax.set_xlim(bounds[0] - pad_x, bounds[2] + pad_x)
    ax.set_ylim(bounds[1] - pad_y, bounds[3] + pad_y)

    try:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldStreetMap,
                        attribution=False, zoom="auto")
    except Exception as e:
        print(f"{city}: basemap fetch failed ({e}); leaving panel blank.")

    aoi.boundary.plot(ax=ax, color=color, linewidth=2.6, zorder=5)

    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f"{city}\n{sub}", fontsize=13, fontweight="bold", color=INK, pad=10)

    # Map-style legend swatch, bottom-right, matching the reference image.
    leg_x, leg_y, leg_w, leg_h = 0.42, 0.03, 0.56, 0.115
    ax.add_patch(Rectangle((leg_x, leg_y), leg_w, leg_h, transform=ax.transAxes,
                            facecolor="white", edgecolor="#888888", linewidth=0.8,
                            zorder=6))
    swatch_x, swatch_y = leg_x + 0.05, leg_y + leg_h / 2
    ax.add_patch(Rectangle((swatch_x, swatch_y - 0.028), 0.14, 0.056,
                            transform=ax.transAxes, facecolor="none",
                            edgecolor=color, linewidth=2.6, zorder=7))
    ax.text(swatch_x + 0.19, swatch_y, f"{city} Study Area", transform=ax.transAxes,
             fontsize=10.5, fontweight="bold", color=INK, va="center", ha="left",
             zorder=7)

fig.tight_layout()
fig.savefig(OUT, dpi=180, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved", OUT)

## 1e. Process slides (stratified sample/split, and RF training + holdout)

In [ ]:
# -*- coding: utf-8 -*-
"""Process slide 1: the stratified spatial sample and the 1 km-block
train/test split, plotted from the real Milan point geometries
(outputs_v2/spatial_{train,test}_pts.gpkg -- 2,449 train / 1,014 test,
matching data/FACTS.md and data/EXPERIMENT_MAP.md exactly). House style
per data/FIGURES.md: DejaVu Sans, INK/ACCENT palette, no top/right
spines, dpi=200.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import geopandas as gpd

MATEJ = r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious\IMD"
OUT = r"C:\Users\user\projects\IMD-Mapping\figs_deck"

INK = "#1a1a1a"
MUTED = "#6b6b6b"
ACCENT = "#0B6E4F"
TEST_C = "#C2724A"
GRID_C = "#ececec"

plt.rcParams.update({"font.family": "DejaVu Sans"})

tr = gpd.read_file(MATEJ + r"\outputs_v2\spatial_train_pts.gpkg")
te = gpd.read_file(MATEJ + r"\outputs_v2\spatial_test_pts.gpkg")
assert len(tr) == 2449 and len(te) == 1014, (len(tr), len(te))

tr = tr.to_crs(32632)
te = te.to_crs(32632)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.1), gridspec_kw={"width_ratios": [1.35, 1]})
ax0, ax1 = axes

ax0.scatter(tr.geometry.x, tr.geometry.y, s=5, color=ACCENT, alpha=0.55,
            linewidths=0, label=f"Train (n={len(tr)})")
ax0.scatter(te.geometry.x, te.geometry.y, s=7, color=TEST_C, alpha=0.75,
            linewidths=0, label=f"Test (n={len(te)})")
ax0.set_title("1 km-block spatial split, Milan", fontsize=12, fontweight="bold", color=INK)
ax0.set_xlabel("Easting (m, UTM 32N)", fontsize=9.5, color=MUTED)
ax0.set_ylabel("Northing (m, UTM 32N)", fontsize=9.5, color=MUTED)
ax0.ticklabel_format(style="plain")
ax0.tick_params(labelsize=8, colors=MUTED)
ax0.set_aspect("equal")
for sp in ("top", "right"):
    ax0.spines[sp].set_visible(False)
ax0.grid(color=GRID_C, linewidth=0.6)
ax0.set_axisbelow(True)
leg = ax0.legend(loc="upper right", fontsize=9.5, frameon=True, markerscale=2.2,
                  facecolor="white", framealpha=1.0, edgecolor=GRID_C,
                  borderpad=0.6, handletextpad=0.6)
leg.set_zorder(10)

# IMD % range each class was stratified over (report.tex, Sampling section):
# the CLMS reference was classified into seven groups before drawing 500
# points at random within each.
CLASS_RANGES = {0: "0%", 1: "1–20%", 2: "21–40%", 3: "41–60%",
                 4: "61–80%", 5: "81–99%", 6: "100%"}

cls_counts_tr = tr["IMD_class"].value_counts().sort_index()
cls_counts_te = te["IMD_class"].value_counts().sort_index()
classes = sorted(set(cls_counts_tr.index) | set(cls_counts_te.index))
x = range(len(classes))
w = 0.38
ax1.bar([i - w / 2 for i in x], [cls_counts_tr.get(c, 0) for c in classes], width=w,
        color=ACCENT, label="Train")
ax1.bar([i + w / 2 for i in x], [cls_counts_te.get(c, 0) for c in classes], width=w,
        color=TEST_C, label="Test")
ax1.set_title("Class balance, train vs test", fontsize=12, fontweight="bold", color=INK)
ax1.set_xlabel("IMD class (stratification range)", fontsize=9.5, color=MUTED)
ax1.set_ylabel("Point count", fontsize=9.5, color=MUTED)
ax1.set_xticks(list(x))
ax1.set_xticklabels([f"C{c}\n{CLASS_RANGES[c]}" for c in classes], fontsize=8.3)
ax1.tick_params(labelsize=8, colors=MUTED)
for sp in ("top", "right"):
    ax1.spines[sp].set_visible(False)
ax1.grid(color=GRID_C, linewidth=0.6, axis="y")
ax1.set_axisbelow(True)
max_count = max(cls_counts_tr.max(), cls_counts_te.max())
ax1.set_ylim(0, max_count * 1.22)
leg1 = ax1.legend(fontsize=9.5, frameon=True, facecolor="white", framealpha=1.0,
                   edgecolor=GRID_C, loc="upper right")
leg1.set_zorder(10)

fig.tight_layout()
fig.savefig(f"{OUT}/process_sampling_split.png", dpi=200, bbox_inches="tight",
            facecolor="white")
plt.close(fig)
print("saved process_sampling_split.png")


# -*- coding: utf-8 -*-
"""Process slide 2: what the random forest learned and how it scored on
the spatial holdout. Left panel -- real permutation feature importance,
Sentinel-2 percentile composite run (data/FACTS.md Table D). Right panel
-- real observed-vs-predicted holdout points (RF only; MLP is out of
scope per CLAUDE.md and is never read), Sentinel-2 stack composite run,
annotated with the exact FACTS.md Table A metrics for that row (RMSE
10.864, MAE 7.641, R2 0.904, Bias -0.169, n=1014). House style per
data/FIGURES.md.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

MATEJ = r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious\IMD"
OUT = r"C:\Users\user\projects\IMD-Mapping\figs_deck"

INK = "#1a1a1a"
MUTED = "#6b6b6b"
ACCENT = "#0B6E4F"
BLUE = "#2E6F9E"
GRID_C = "#ececec"

plt.rcParams.update({"font.family": "DejaVu Sans"})

fi = pd.read_csv(MATEJ + r"\outputs_S2_percentile_p10p25p50p75p90\feature_importance_RF.csv")
fi_top = fi.sort_values("rank").head(6)

hr = pd.read_csv(MATEJ + r"\outputs_S2_stack\holdout_residuals.csv")
assert len(hr) == 1014

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.0, 5.1))

bars = ax0.barh(fi_top["band"][::-1], fi_top["perm_importance"][::-1], color=ACCENT)
ax0.set_title("Top predictor bands by permutation importance\n(Sentinel-2 percentile composite run)",
               fontsize=11.5, fontweight="bold", color=INK)
ax0.set_xlabel("Permutation importance", fontsize=9.5, color=MUTED)
ax0.tick_params(labelsize=9.5, colors=MUTED)
for sp in ("top", "right"):
    ax0.spines[sp].set_visible(False)
ax0.grid(color=GRID_C, linewidth=0.6, axis="x")
ax0.set_axisbelow(True)
for b, v in zip(bars, fi_top["perm_importance"][::-1]):
    ax0.text(v + 0.003, b.get_y() + b.get_height() / 2, f"{v:.3f}",
              va="center", fontsize=9, color=INK)

ax1.scatter(hr["IMD"], hr["pred_RF"], s=10, color=BLUE, alpha=0.35, linewidths=0)
ax1.plot([0, 100], [0, 100], color=MUTED, linewidth=1.1, linestyle="--")
ax1.set_xlim(-2, 102)
ax1.set_ylim(-2, 102)
ax1.set_title("Holdout: observed vs predicted IMD\n(Sentinel-2 stack composite run, random forest)",
               fontsize=11.5, fontweight="bold", color=INK)
ax1.set_xlabel("Observed IMD (%)", fontsize=9.5, color=MUTED)
ax1.set_ylabel("Predicted IMD (%)", fontsize=9.5, color=MUTED)
ax1.tick_params(labelsize=9, colors=MUTED)
ax1.set_aspect("equal")
for sp in ("top", "right"):
    ax1.spines[sp].set_visible(False)
ax1.grid(color=GRID_C, linewidth=0.6)
ax1.set_axisbelow(True)
metrics_txt = "RMSE 10.864\nMAE 7.641\nR\u00b2 0.904\nBias \u22120.169\nn = 1014"
ax1.text(0.04, 0.96, metrics_txt, transform=ax1.transAxes, fontsize=9.5, color=INK,
          va="top", ha="left",
          bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=GRID_C))

fig.tight_layout()
fig.savefig(f"{OUT}/process_training.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved process_training.png")


## 2. Per-city results grids (predicted maps + difference against benchmark)

In [ ]:
# -*- coding: utf-8 -*-
"""Per-city results grids: predicted maps (+ benchmark) on one row, the
difference against that benchmark on the row below. Milan gets two such
row-pairs (CLMS-trained, then GHSL-trained); Hanoi and HCMC get one each
(against GHS-BUILT-S). Saved to figs_deck/.
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

MATEJ = Path(r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious")
REPO = Path(r"C:\Users\user\projects\IMD-Mapping")
OUT = REPO / "figs_deck"

IMD_CMAP = ListedColormap(["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"])
THUMB = 340


def read_thumb(path, clip_to=None, out_shape=None):
    with rasterio.open(path) as src:
        if clip_to is not None:
            with rasterio.open(clip_to) as ref:
                b = ref.bounds
            win = from_bounds(*b, transform=src.transform)
            h, w = win.height, win.width
            if out_shape is None:
                scale = max(h, w) / THUMB
                out_shape = (max(1, int(h / scale)), max(1, int(w / scale)))
            arr = src.read(1, window=win, out_shape=(1,) + out_shape,
                            resampling=Resampling.average, boundless=True)
            extent = (b.left, b.right, b.bottom, b.top)
        else:
            h, w = src.height, src.width
            if out_shape is None:
                scale = max(h, w) / THUMB
                out_shape = (max(1, int(h / scale)), max(1, int(w / scale)))
            arr = src.read(1, out_shape=(1,) + out_shape, resampling=Resampling.average)
            b = src.bounds
            extent = (b.left, b.right, b.bottom, b.top)
        nodata = src.nodata
        arr = arr.astype("float32")
        if nodata is not None:
            arr[arr == nodata] = np.nan
        arr[(arr < 0) | (arr > 100)] = np.nan
    return arr, extent, out_shape


def panel(ax, arr, extent, cmap, vmin, vmax, title):
    im = ax.imshow(arr, extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, fontsize=10.5, pad=4)
    return im


def build_grid(out_name, row_groups, ncols=5, figsize=(18, None), group_labels=None):
    """row_groups: list of (pred_row, diff_row) pairs.
    pred_row: list of (title, path, clip_to_or_None) -- IMD_CMAP, 0..100.
    diff_row: list of (title, pred_path, bench_path, clip_to_or_None) -- RdBu_r, -30..30.
    group_labels: one label per row_group, printed rotated in the left
    margin -- makes each row-pair self-identifying (what it is trained on
    and scored against) without needing slide text. Only useful with more
    than one row_group; omit for a single one.
    """
    nrows = 2 * len(row_groups)
    h_per_row = 3.1
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize[0], h_per_row * nrows))
    if nrows == 1:
        axes = axes[None, :]
    im_pred = im_diff = None

    for gi, (pred_row, diff_row) in enumerate(row_groups):
        r_pred = 2 * gi
        r_diff = 2 * gi + 1
        for c in range(ncols):
            ax = axes[r_pred, c]
            if c < len(pred_row):
                title, path, clip_to = pred_row[c]
                arr, extent, _ = read_thumb(path, clip_to)
                im_pred = panel(ax, arr, extent, IMD_CMAP, 0, 100, title)
            else:
                ax.axis("off")
        for c in range(ncols):
            ax = axes[r_diff, c]
            if c < len(diff_row):
                title, pred_path, bench_path, clip_to = diff_row[c]
                parr, extent, shp = read_thumb(pred_path, clip_to)
                barr, _, _ = read_thumb(bench_path, clip_to, out_shape=shp)
                diff = parr - barr
                im_diff = panel(ax, diff, extent, "RdBu_r", -30, 30,
                                 title + "\n(minus benchmark)")
            else:
                ax.axis("off")

    left_margin = 0.055 if group_labels else 0.03
    top, bottom = 0.96, 0.02
    fig.subplots_adjust(left=left_margin, right=0.90, top=top, bottom=bottom,
                         wspace=0.08, hspace=0.32)

    if group_labels:
        band = (top - bottom) / nrows
        for gi, label in enumerate(group_labels):
            y0 = top - gi * 2 * band
            ycenter = y0 - band
            fig.text(0.014, ycenter, label, rotation=90, ha="center", va="center",
                      fontsize=12, fontweight="bold", color="#333333")

    if im_pred is not None:
        cax1 = fig.add_axes([0.915, 0.53, 0.013, 0.4])
        fig.colorbar(im_pred, cax=cax1, label="Predicted IMD (%)")
    if im_diff is not None:
        cax2 = fig.add_axes([0.915, 0.06, 0.013, 0.4])
        fig.colorbar(im_diff, cax=cax2, label="Predicted minus benchmark (pp)")

    path = OUT / out_name
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("saved", path)


# =====================================================================
# MILAN: two row-pairs, CLMS-trained then GHSL-trained
# =====================================================================
CLMS = MATEJ / "matej_files_codes/reference_IMD/IMD_2018_CLMS_Milan.tif"
GHSL_PROD = REPO / "data/GHSL_2018_Milan_UTM32N.tif"

milan_clms_models = [
    ("S2 stack", MATEJ / "IMD/outputs_S2_stack/IMD_predicted_RF_S2_Milan.tif"),
    ("S2 percentile", MATEJ / "IMD/outputs_S2_percentile_p10p25p50p75p90/IMD_predicted_RF_S2_Milan.tif"),
    ("S2 median", MATEJ / "IMD/outputs_S2_median/IMD_predicted_RF_S2_Milan.tif"),
    ("AlphaEarth", MATEJ / "IMD/outputs_v2/IMD_predicted_RF_spatialCV2_Milan.tif"),
]
milan_ghsl_models = [
    ("S2 stack \u00b7 GHSL", MATEJ / "IMD/outputs_S2_stack_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("S2 percentile \u00b7 GHSL", MATEJ / "IMD/outputs_S2_percentile_p10p25p50p75p90_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("S2 median \u00b7 GHSL", MATEJ / "IMD/outputs_S2_median_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("AlphaEarth \u00b7 GHSL", MATEJ / "IMD/outputs_GHSL/GHSL_predicted_RF_spatialCV2_Milan.tif"),
]

pred_row1 = [(n, p, None) for n, p in milan_clms_models] + [("CLMS (benchmark)", CLMS, None)]
diff_row1 = [(n, p, CLMS, None) for n, p in milan_clms_models]
pred_row2 = [(n, p, None) for n, p in milan_ghsl_models] + [("GHS-BUILT-S (benchmark)", GHSL_PROD, None)]
diff_row2 = [(n, p, GHSL_PROD, None) for n, p in milan_ghsl_models]

build_grid("results_milan_grid.png", [(pred_row1, diff_row1), (pred_row2, diff_row2)],
           ncols=5, figsize=(18, None),
           group_labels=["Trained on CLMS", "Trained on GHS-BUILT-S"])

# =====================================================================
# HANOI and HCMC: one row-pair each, against GHS-BUILT-S
# =====================================================================
for city, aoi_ref, ghsl_target, models in [
    ("Hanoi", MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_localrf.tif",
     MATEJ / "matej_files_codes/reference_IMD/IMD_2018_Hanoi.tif",
     [("AlphaEarth\n(local retrain)", MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_localrf.tif"),
      ("AlphaEarth\n(zero-shot)", MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_zeroshot.tif"),
      ("S2 median\n(local retrain)", MATEJ / "IMD/outputs_transfer_S2_median/IMD_Hanoi_10m_localrf_S2median.tif"),
      ("S2 median\n(zero-shot)", MATEJ / "IMD/outputs_transfer_S2_median/IMD_Hanoi_10m_zeroshot_S2median.tif")]),
    ("HCMC", MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_localrf.tif",
     MATEJ / "matej_files_codes/reference_IMD/IMD_2018_HCMC.tif",
     [("AlphaEarth\n(local retrain)", MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_localrf.tif"),
      ("AlphaEarth\n(zero-shot)", MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_zeroshot.tif"),
      ("S2 median\n(local retrain)", MATEJ / "IMD/outputs_transfer_S2_median/IMD_HCMC_10m_localrf_S2median.tif"),
      ("S2 median\n(zero-shot)", MATEJ / "IMD/outputs_transfer_S2_median/IMD_HCMC_10m_zeroshot_S2median.tif")]),
]:
    pred_row = [(n, p, None) for n, p in models] + [("GHS-BUILT-S (benchmark)", ghsl_target, aoi_ref)]
    diff_row = [(n, p, ghsl_target, aoi_ref) for n, p in models]
    build_grid(f"results_{city.lower()}_grid.png", [(pred_row, diff_row)],
               ncols=5, figsize=(18, None))

print("done.")

## 3. Assemble the deck on the Polimi template

In [ ]:
# -*- coding: utf-8 -*-
"""Build the results-overview deck: Introduction, Study area, Data, Methodology
(pipeline diagram), Results (per-city map grids + per-city 3-technique
validation), Conclusion. Polimi template, same helper pattern as
build_gisideas_deck.ipynb.
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

import re

from PIL import Image
from pptx import Presentation
from pptx.util import Emu, Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor

TEMPLATE = (r"C:\Users\user\OneDrive - Politecnico di Milano\Keerthana_phd_folder"
            r"\Annual report\Template_ppt_2024_DICA_v1.pptx")
OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_Vietnam_presentation.pptx")
FIGS = r"C:\Users\user\projects\IMD-Mapping\figs_deck"

DATE_TXT = "IMD Mapping  \u00b7  Milan, Hanoi, Ho Chi Minh City"
FOOTER_TXT = "Impervious Surface Density from Sentinel-2 and AlphaEarth"

MUTED_RGB = RGBColor(0x40, 0x40, 0x40)
NOTE_RGB = RGBColor(0x60, 0x60, 0x60)

prs = Presentation(TEMPLATE)
M0, M1 = prs.slide_masters[0], prs.slide_masters[1]


def layout(master, name):
    for lay in master.slide_layouts:
        if lay.name == name:
            return lay
    raise KeyError(name)


L_TITLE = layout(M1, "Cover G")
L_AGENDA = layout(M0, "Indice_A")
L_DIVIDER = layout(M0, "Divisorio_A")
L_TEXT = layout(M0, "Pagina base_ testo A")
L_TEXT2 = layout(M0, "Pagina base_ testo B")
L_TEXT3 = layout(M0, "Pagina grafica _ 3 colonne")
L_TABLE = layout(M0, "Pagina base_Oggetto")
L_IMAGE = layout(M0, "Pagina immagine _ A")
L_FINALE = layout(M0, "Slide_finale_B")
L_CONTACT = layout(M1, "Cover Finale")

xml_slides = prs.slides._sldIdLst
for sld_id in list(xml_slides):
    prs.part.drop_rel(sld_id.get(
        "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"))
    xml_slides.remove(sld_id)


def add(lay):
    return prs.slides.add_slide(lay)


def meta(slide, date=DATE_TXT, footer=FOOTER_TXT):
    idxs = [p.placeholder_format.idx for p in slide.placeholders]
    for idx, val in ((10, date), (11, footer)):
        if idx in idxs:
            slide.placeholders[idx].text = val


# ---- rich-text helpers -----------------------------------------------------
# Every text helper below understands **bold** markdown so an important word
# or number can be highlighted inline without hand-building runs at each
# call site. Plain text with no "**" renders exactly as it did before.
_BOLD_RE = re.compile(r"(\*\*.*?\*\*)")


def _add_rich_runs(paragraph, text, size=None, color=None, italic=None, bold_base=False):
    for part in _BOLD_RE.split(text):
        if part == "":
            continue
        is_bold = part.startswith("**") and part.endswith("**")
        run = paragraph.add_run()
        run.text = part[2:-2] if is_bold else part
        run.font.bold = True if is_bold else bold_base
        if size is not None:
            run.font.size = Pt(size)
        if italic is not None:
            run.font.italic = italic
        if color is not None:
            run.font.color.rgb = color
    return paragraph


def settxt(slide, idx, text):
    """Title/label text. Supports **bold** so a key term can stand out even
    in a short label; plain strings render exactly as slide.placeholders.text
    used to. tf.clear() first -- a freshly added placeholder can carry a
    prompt run of its own, and without clearing, our new runs would just
    append after it instead of replacing it."""
    tf = slide.placeholders[idx].text_frame
    tf.clear()
    _add_rich_runs(tf.paragraphs[0], text)


def fill_body(placeholder, lines):
    """Bulleted body text. Each line may contain **bold** spans."""
    tf = placeholder.text_frame
    tf.clear()
    tf.word_wrap = True
    for i, line in enumerate(lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        _add_rich_runs(p, line)


def add_rich_textbox(slide, x, y, w, h, lines, size=10.5, color=None, italic=None,
                      bullet=None, space_after=None):
    """A free-standing textbox of rich-text lines/paragraphs, used for the
    custom (non-placeholder) content on the map, validation and study-area
    slides."""
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        if space_after is not None and i > 0:
            p.space_before = Pt(space_after)
        if bullet:
            r = p.add_run()
            r.text = bullet
            r.font.size = Pt(size)
            if color is not None:
                r.font.color.rgb = color
        _add_rich_runs(p, line, size=size, color=color, italic=italic)
    return tb


def add_table(slide, idx, headers, rows, font_size=11):
    ph = slide.placeholders[idx]
    x, y, w, h = ph.left, ph.top, ph.width, ph.height
    ph._element.getparent().remove(ph._element)
    gframe = slide.shapes.add_table(len(rows) + 1, len(headers), x, y, w, h)
    table = gframe.table
    for j, hd in enumerate(headers):
        table.cell(0, j).text = str(hd)
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            table.cell(i + 1, j).text = str(v)
    for r in table.rows:
        for c in r.cells:
            for p in c.text_frame.paragraphs:
                for run in p.runs:
                    run.font.size = Pt(font_size)
    return table


def add_mini_table(slide, x, y, w, headers, rows, col_widths, font_size=9.5, row_h=0.30):
    n_rows = len(rows) + 1
    gframe = slide.shapes.add_table(n_rows, len(headers), x, y, w, Inches(row_h * n_rows))
    table = gframe.table
    for j, cw in enumerate(col_widths):
        table.columns[j].width = Inches(cw)
    for j, hd in enumerate(headers):
        table.cell(0, j).text = str(hd)
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            table.cell(i + 1, j).text = str(v)
    for i, r in enumerate(table.rows):
        r.height = Inches(row_h)
        for c in r.cells:
            for p in c.text_frame.paragraphs:
                for run in p.runs:
                    run.font.size = Pt(font_size)
                    run.font.bold = (i == 0)
    return table


def add_picture_fit(slide, path, x, y, w, h):
    iw, ih = Image.open(path).size
    ar = iw / ih
    box_ar = w / h
    if ar > box_ar:
        pw, ph = w, Emu(int(w / ar))
    else:
        ph, pw = h, Emu(int(h * ar))
    px = x + Emu(int((w - pw) / 2))
    py = y + Emu(int((h - ph) / 2))
    slide.shapes.add_picture(path, px, py, width=pw, height=ph)


def divider(title, number):
    s = add(L_DIVIDER)
    settxt(s, 0, title)
    meta(s)
    settxt(s, 13, number)
    return s


def text_slide(title, lines):
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s)
    fill_body(s.placeholders[13], lines)
    return s


def text2_slide(title, head1, lines1, head2, lines2):
    s = add(L_TEXT2)
    settxt(s, 0, title)
    meta(s)
    fill_body(s.placeholders[13], [head1, ""] + lines1)
    fill_body(s.placeholders[14], [head2, ""] + lines2)
    return s


def text3_slide(title, cols):
    """cols: list of (head, lines) triples, exactly 3."""
    s = add(L_TEXT3)
    settxt(s, 0, title)
    meta(s)
    idx_pairs = [(13, 15), (16, 17), (18, 19)]
    for (head, lines), (hidx, bidx) in zip(cols, idx_pairs):
        # hidx is the tall lower box, bidx the short box right under the
        # title -- head text (short) goes in bidx, body lines (long) in
        # hidx, or the long lines overflow upward into the title.
        settxt(s, bidx, head)
        fill_body(s.placeholders[hidx], lines)
    return s


def table_slide(title, note, headers, rows, font_size=11):
    s = add(L_TABLE)
    settxt(s, 0, title)
    meta(s)
    add_table(s, 1, headers, rows, font_size=font_size)
    settxt(s, 16, note)
    return s


def big_image_slide(title, img_path, subtitle_lines=None, note=None, note_h=0.6):
    """note: one or more rich-text lines shown under the image, bulleted
    when there is more than one. note_h (inches) reserves room for them --
    the image shrinks by exactly that much, so every existing single-line
    caller (note_h defaults to 0.6, matching what they already looked
    like) is unaffected. subtitle_lines (text above the image instead)
    and note are mutually exclusive."""
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s)
    body = s.placeholders[13]
    if subtitle_lines:
        fill_body(body, subtitle_lines)
        y0 = Inches(3.05)
        h0 = Inches(4.15)
    else:
        body._element.getparent().remove(body._element)
        if note:
            y0 = Inches(1.85)
            h0 = Inches(5.35 - note_h)
        else:
            y0 = Inches(2.02)
            h0 = Inches(5.15)
    add_picture_fit(s, img_path, Inches(0.2), y0, Inches(12.95), h0)
    if note:
        note_lines = note if isinstance(note, list) else [note]
        y_note = y0 + h0 + Inches(0.12)
        add_rich_textbox(s, Inches(0.5), y_note, Inches(12.35), Inches(note_h),
                          note_lines, size=12, color=RGBColor(0x1a, 0x1a, 0x1a),
                          bullet=("\u2022 " if len(note_lines) > 1 else None))
    return s


# ===================================================================== 1. title
s = add(L_TITLE)
settxt(s, 22, "Impervious Surface Density Mapping, 2018")
settxt(s, 0, "Mapping Imperviousness Density Using Geospatial\n"
             "Foundation-Model AlphaEarth Embeddings")
settxt(s, 21, "Keerthana Kirubakaran, Xiao Tan  \u00b7  Politecnico di Milano")

# ===================================================================== 2. agenda
s = add(L_AGENDA)
settxt(s, 0, "Outline")
meta(s)
fill_body(s.placeholders[13], ["Milan \u00b7 Hanoi \u00b7 HCMC", "2018 \u00b7 10 m"])
fill_body(s.placeholders[15], [
    "1.  Introduction",
    "2.  Study area and the importance of impervious mapping",
    "3.  Data used",
    "4.  Methodology",
    "5.  Results: predicted maps and independent validation",
    "6.  Conclusion and key findings",
])

# ===================================================================== 3. divider 01
divider("Introduction", "01")

# ===================================================================== 4. introduction
big_image_slide(
    "Impervious surface: what it is, why it is measured",
    f"{FIGS}\\imd_concept_full.png",
    note=[
        "Sealed ground cannot absorb rain or cool by evaporation, which drives **urban flooding** and the **urban heat island**.",
        "IMD feeds directly into how neighbourhoods are classified for climate risk (**LCZ**).",
        "Mapped here at **10 m** for **Milan**, **Hanoi** and **Ho Chi Minh City**, **2018**, "
        "comparing **AlphaEarth** against **Sentinel-2**.",
    ],
    note_h=1.05,
)

# ===================================================================== 5. divider 02
divider("Study area", "02")

# ============================================================= 6. study area (merged)
def study_area_slide(title, intro, img_path, city_facts, closing):
    """One slide replacing the old three: a short framing line, the world
    map (with its own insets), then one condensed line per city, then a
    closing line on why these three cities. city_facts: list of (name, text)."""
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s)
    body = s.placeholders[13]
    body._element.getparent().remove(body._element)

    add_rich_textbox(s, Inches(0.4), Inches(1.68), Inches(12.5), Inches(0.4),
                      [intro], size=12.5, color=RGBColor(0x1a, 0x1a, 0x1a))

    add_picture_fit(s, img_path, Inches(2.15), Inches(2.12), Inches(9.05), Inches(3.95))

    y0 = Inches(6.20)
    col_w = Inches(4.05)
    gap = Inches(0.12)
    x1 = Inches(0.37)
    x2 = x1 + col_w + gap
    x3 = x2 + col_w + gap
    for (name, fact), x in zip(city_facts, (x1, x2, x3)):
        add_rich_textbox(s, x, y0, col_w, Inches(0.80),
                          [f"**{name}** \u2014 {fact}"], size=9.5,
                          color=RGBColor(0x2a, 0x2a, 0x2a))

    add_rich_textbox(s, Inches(0.37), Inches(7.14), Inches(12.6), Inches(0.32),
                      [closing], size=9.3, italic=True, color=NOTE_RGB)
    return s


study_area_slide(
    "Three cities, one shared question",
    ("Milan, Italy and Hanoi and Ho Chi Minh City, Vietnam **~8,900 km apart**, "
     "test whether one impervious-mapping method **generalises** across climate, "
     "data and urban form."),
    f"{FIGS}\\study_area_world_map.png",
    [
        ("Milan, Italy",
         "**1.4M** city, **3M** metro, temperate climate; impervious cover drives summer "
         "**heat island** intensity and flood risk. The only city with a **CLMS** reference, "
         "alongside **GHS-BUILT-S**."),
        ("Hanoi, Vietnam",
         "**8M+** residents on the Red River delta, monsoon climate; impervious expansion "
         "directly raises **flash-flood risk**. No fine-grained product existed before this "
         "study beyond **GHS-BUILT-S**."),
        ("Ho Chi Minh City, Vietnam",
         "**9M+** residents, low-lying delta terrain, among the most **flood-exposed** cities "
         "in Southeast Asia. Tests whether a model trained elsewhere transfers with "
         "**zero local training data**."),
    ],
    ("Supports the Italy-Vietnam **LCZ-UHI-GEO** project; three climates test whether one "
     "method generalises across cities."),
)

# ===================================================================== 7. divider 03
divider("Data used", "03")

# ===================================================================== 8. data table
table_slide(
    "Five data sets, three roles",
    "Note: **AlphaEarth** and **Sentinel-2** are the predictors throughout. Sentinel-2 "
    "composites use **10 spectral bands**: **4 native at 10 m** (B2, B3, B4, B8) and "
    "**6 native at 20 m** (B5, B6, B7, B8A, B11, B12), all resampled to **10 m**. "
    "**CLMS** and **GHS-BUILT-S** are training targets and, scored independently, maps "
    "under test. **EarthLabel** is never used in training.",
    ["Data set", "Type", "Resolution", "Coverage", "Role"],
    [["AlphaEarth embeddings", "64-band annual satellite embedding", "10 m",
      "Milan, Hanoi, HCMC", "Predictor"],
     ["Sentinel-2 composites", "Median / stack / percentile reflectance", "10 m output "
      "(10 + 20 m bands)", "Milan 30 dates, Hanoi 4, HCMC 3", "Predictor"],
     ["CLMS Imperviousness Density", "Sealed surface, 2018", "10 m",
      "Milan only (Europe)", "Training target, Milan"],
     ["GHS-BUILT-S", "Built-up surface, 2018", "10 m",
      "Milan, Hanoi, HCMC (global)", "Training target, all cities"],
     ["EarthLabel plots", "Photo-interpreted, 2018 imagery", "10 m per plot",
      "450 plots per city, 1,350 total", "Independent validation only"]],
    font_size=11,
)

# ===================================================================== 9. divider 04
divider("Methodology", "04")

# ===================================================================== 10. methodology
big_image_slide(
    "From five data sources to one independent validation",
    f"{FIGS}\\methodology_pipeline.png",
)

# ---- process: sampling/split, then training + holdout accuracy
big_image_slide(
    "Stratified spatial sampling and the 1 km-block split",
    f"{FIGS}\\process_sampling_split.png",
    note=[
        "**500 points per IMD class** (3,500 total) sampled across Milan, then assigned by "
        "**whole 1 km block** to train or test so nearby points never leak across the split.",
        "A **250 m buffer** removes any block edge-adjacent to the other set, leaving "
        "**2,449 train / 1,014 test** points, balanced across all seven classes.",
    ],
    note_h=1.05,
)

big_image_slide(
    "Training the random forest, and what it learned",
    f"{FIGS}\\process_training.png",
    note=[
        "The **low percentiles of red (B4) and high percentiles of near-infrared (B8)** "
        "dominate feature importance \u2014 exactly the seasonal extremes a single-date or "
        "median composite discards.",
        "On the **1,014-point spatial holdout**, the trained model reaches **RMSE 10.86, "
        "R\u00b2 0.90** \u2014 accuracy the raster prediction step then carries onto the full scene.",
    ],
    note_h=1.05,
)

# ===================================================================== 11. divider 05
divider("Results", "05")

# ---- Milan maps
big_image_slide(
    "Milan: predicted maps against both benchmarks",
    f"{FIGS}\\results_milan_grid.png",
    note=["The **GHS-BUILT-S-trained row flattens** toward that product's muted style, and "
          "reads **redder** in its difference row: these models still predict the roads the "
          "training label omits."],
)

# ---- Hanoi maps
big_image_slide(
    "Hanoi: predicted maps, zero-shot against local retrain",
    f"{FIGS}\\results_hanoi_grid.png",
    note=["**Zero-shot transfer fails almost everywhere** (dark red, over **20 points high**); "
          "**local retraining brings the error back near zero** across most of the scene."],
)

# ---- HCMC maps
big_image_slide(
    "Ho Chi Minh City: predicted maps, zero-shot against local retrain",
    f"{FIGS}\\results_hcmc_grid.png",
    note=["Same pattern as Hanoi, but sharper: the **AlphaEarth zero-shot map is the most "
          "over-predicted raster in the whole study**, saturated red almost edge to edge."],
)

# ---- why three validation techniques (once, before the per-city tables)
def why_three_slide(title, intro, panels):
    """panels: list of (label, lines) triples. Custom layout (not text3_slide)
    so a short rationale line can sit between the title and the three
    technique explanations."""
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s)
    body = s.placeholders[13]
    body._element.getparent().remove(body._element)

    add_rich_textbox(s, Inches(0.4), Inches(1.68), Inches(12.5), Inches(0.75),
                      [intro], size=12.5, color=RGBColor(0x1a, 0x1a, 0x1a))

    col_w = Inches(4.05)
    gap = Inches(0.12)
    x1 = Inches(0.37)
    x2 = x1 + col_w + gap
    x3 = x2 + col_w + gap
    y_label = Inches(2.75)
    y_body = Inches(3.15)
    for (label, lines), x in zip(panels, (x1, x2, x3)):
        tb = s.shapes.add_textbox(x, y_label, col_w, Inches(0.35))
        p = tb.text_frame.paragraphs[0]
        _add_rich_runs(p, label, size=12.5, bold_base=True)
        add_rich_textbox(s, x, y_body, col_w, Inches(2.4), lines, size=10.8,
                          color=RGBColor(0x2a, 0x2a, 0x2a), space_after=6)
    return s


why_three_slide(
    "Why three validation techniques",
    ("**IMD gets used three different ways once it leaves this study**: as a raw percentage, "
     "as a sealed / not-sealed threshold for flood and heat-island screening, and as an "
     "ordinal scale where being one class off matters far less than being ten classes off. "
     "**A model can win on one of these and still fail the other two** \u2014 so no single "
     "technique is enough to call a raster the best, and every map here is checked against "
     "all three, against the same **450 EarthLabel plots** per city."),
    [
        ("A \u00b7 Continuous error",
         ["RMSE, MAE and bias, in IMD percentage points.",
          "**Chosen because** IMD is reported as a percentage \u2014 this is the most direct "
          "read of raw accuracy, and the only one of the three that shows over- vs "
          "under-prediction."]),
        ("B \u00b7 Binary agreement",
         ["Overall accuracy and Cohen's kappa at a **50%** sealed / not-sealed cut.",
          "**Chosen because** flood and heat-island screening often only needs a yes/no "
          "sealed call, and kappa corrects for how often that call is right by chance alone."]),
        ("C \u00b7 Ordinal agreement",
         ["Overall accuracy and quadratic weighted kappa across the full **10-class** scale.",
          "**Chosen because** a map can post a good RMSE while still reading one class off "
          "everywhere, or a bad RMSE from a few wild misses \u2014 QWK is what tells those "
          "two failure modes apart."]),
    ],
)


def val_slide(city, rows, points):
    """rows: [raster, RMSE, MAE, Bias, OA_b, kappa, OA_c, QWK] per raster,
    already sorted best-to-worst by continuous error. Rendered as three
    small tables, one per validation technique, sharing that row order."""
    s = add(L_TEXT)
    settxt(s, 0, f"{city}: three validation techniques")
    meta(s)
    body = s.placeholders[13]
    body._element.getparent().remove(body._element)

    n = len(rows)
    row_h = 0.30 if n > 6 else 0.42
    fs = 9.3 if n > 6 else 10.8

    col_w = Inches(4.10)
    gap = Inches(0.15)
    x1 = Inches(0.37)
    x2 = x1 + col_w + gap
    x3 = x2 + col_w + gap
    y_head = Inches(1.78)
    y_tab = Inches(2.15)

    panels = [
        ("A \u00b7 continuous error", ["Raster", "RMSE", "MAE", "Bias"], [1.55, 0.85, 0.85, 0.85],
         [[r[0], r[1], r[2], r[3]] for r in rows]),
        ("B \u00b7 binary agreement (50% cut)", ["Raster", "OA", "kappa"], [2.30, 0.90, 0.90],
         [[r[0], r[4], r[5]] for r in rows]),
        ("C \u00b7 ordinal agreement (10-class)", ["Raster", "OA", "QWK"], [2.30, 0.90, 0.90],
         [[r[0], r[6], r[7]] for r in rows]),
    ]
    for (label, headers, col_widths, table_rows), x in zip(panels, (x1, x2, x3)):
        tb = s.shapes.add_textbox(x, y_head, col_w, Inches(0.3))
        p = tb.text_frame.paragraphs[0]
        _add_rich_runs(p, label, size=10.8, bold_base=True)
        add_mini_table(s, x, y_tab, col_w, headers, table_rows, col_widths,
                        font_size=fs, row_h=row_h)

    y_note = y_tab + Inches(row_h * (n + 1)) + Inches(0.12)
    add_rich_textbox(
        s, Inches(0.37), y_note, Inches(12.6), Inches(0.85),
        ["**RMSE** and **MAE** average the error, in IMD percentage points, between "
         "predicted and observed; RMSE penalises large misses more. **Bias** is observed "
         "minus predicted, so positive means the map **under-predicts**. **OA** is the "
         "plain share of the 450 plots the map gets right; **kappa** corrects that for "
         "chance agreement at the 50% cut. **QWK** extends that logic to the full 10-class "
         "scale, penalising a far-off miss more than an adjacent-class one."],
        size=9.3, italic=True, color=NOTE_RGB,
    )

    y_pts = y_note + Inches(0.92)
    add_rich_textbox(s, Inches(0.37), y_pts, Inches(12.6), Inches(1.0), points,
                      size=10.3, color=MUTED_RGB, bullet="\u2022 ")
    return s


val_slide(
    "Milan",
    [["S2 stack", "24.56", "16.53", "\u22123.32", "0.851", "0.692", "0.349", "0.793"],
     ["S2 percentile", "24.73", "16.46", "\u22123.32", "0.838", "0.666", "0.367", "0.788"],
     ["AlphaEarth", "25.56", "18.27", "\u22121.60", "0.820", "0.627", "0.311", "0.760"],
     ["S2 median", "25.91", "18.34", "\u22124.15", "0.840", "0.670", "0.240", "0.760"],
     ["CLMS (benchmark)", "26.23", "14.83", "2.24", "0.824", "0.635", "0.527", "0.779"],
     ["S2 percentile \u00b7 GHSL", "28.23", "19.92", "9.46", "0.784", "0.526", "0.387", "0.696"],
     ["S2 stack \u00b7 GHSL", "28.38", "20.45", "9.19", "0.767", "0.483", "0.364", "0.687"],
     ["AlphaEarth \u00b7 GHSL", "28.92", "21.00", "10.02", "0.762", "0.471", "0.327", "0.665"],
     ["S2 median \u00b7 GHSL", "29.25", "20.95", "9.26", "0.767", "0.484", "0.351", "0.668"],
     ["GHS-BUILT-S (benchmark)", "36.81", "24.21", "18.97", "0.702", "0.312", "0.433", "0.504"]],
    ["**S2 stack** wins on all three techniques \u2014 the training label matters more than "
     "Sentinel-2 vs AlphaEarth ever does.",
     "Every **CLMS-trained** map beats every **GHS-BUILT-S-trained** map, on all three "
     "techniques, without exception.",
     "Bias flips between the two groups: CLMS-trained models read slightly high; "
     "GHS-BUILT-S-trained models inherit part of that product's under-prediction."],
)

val_slide(
    "Hanoi",
    [["AlphaEarth (local retrain)", "26.50", "19.59", "6.59", "0.864", "0.726", "0.309", "0.760"],
     ["S2 median (local retrain)", "27.28", "21.32", "8.18", "0.864", "0.725", "0.240", "0.741"],
     ["S2 median (zero-shot)", "33.79", "24.47", "\u221215.88", "0.804", "0.615", "0.249", "0.631"],
     ["AlphaEarth (zero-shot)", "34.35", "29.54", "\u221215.72", "0.796", "0.599", "0.042", "0.561"],
     ["GHS-BUILT-S (benchmark)", "40.35", "27.05", "19.68", "0.700", "0.381", "0.389", "0.501"]],
    ["**Local retraining wins, zero-shot transfer loses** \u2014 on all three techniques, no "
     "exceptions.",
     "GHS-BUILT-S itself finishes **last** on every technique, behind every model including "
     "the zero-shot ones it under-marks against.",
     "**AlphaEarth (local retrain)** is Hanoi's best raster overall \u2014 the only city here "
     "where AlphaEarth, not Sentinel-2, tops the ranking."],
)

val_slide(
    "Ho Chi Minh City",
    [["S2 median (zero-shot)", "25.95", "16.78", "\u22126.83", "0.856", "0.710", "0.389", "0.803"],
     ["AlphaEarth (local retrain)", "27.24", "20.29", "11.09", "0.840", "0.682", "0.276", "0.750"],
     ["S2 median (local retrain)", "29.62", "23.03", "12.23", "0.813", "0.630", "0.202", "0.703"],
     ["GHS-BUILT-S (benchmark)", "37.35", "25.73", "19.80", "0.729", "0.466", "0.320", "0.590"],
     ["AlphaEarth (zero-shot)", "38.49", "32.40", "\u221219.64", "0.691", "0.366", "0.049", "0.448"]],
    ["**S2 median (zero-shot)** wins on all three techniques \u2014 the one city here where "
     "zero-shot transfer actually leads.",
     "That is **level matching, not successful transfer**: Milan's predicted level happens "
     "to sit close to HCMC's reference level, by coincidence of the two cities' own "
     "averages.",
     "**AlphaEarth (zero-shot)** is HCMC's worst raster on every technique \u2014 though the "
     "raw GHS-BUILT-S product reads even lower elsewhere in the study."],
)

# ===================================================================== conclusion divider
divider("Conclusion", "06")

# ============================================================ conclusion, narrative
text_slide("What this study concludes", [
    "**AlphaEarth** and **Sentinel-2** map impervious surface almost equally well, as long "
    "as both are trained locally.",
    "Zero-shot transfer fails outright. A model really only works well in the city it was "
    "trained in.",
    "Local retraining fixes most of that failure, though not all of it.",
    "Only an **independent reference** tells you real accuracy. Every training product, "
    "checked that way, is beaten by the model fitted to it.",
])

# ===================================================================== key findings
text2_slide(
    "Key findings",
    "What the maps and the numbers agree on",
    ["**Sentinel-2** beats **AlphaEarth** in Milan, by a wide margin, when both are checked "
     "against CLMS.",
     "The training label matters as much as the feature set. Swapping Milan's label to "
     "GHS-BUILT-S costs about **4 RMSE**.",
     "Zero-shot transfer to Vietnam fails outright. Every difference map turns solid red.",
     "Local retraining recovers **38 to 67%** of GHS-BUILT-S's own systematic "
     "under-prediction."],
    "What to take forward",
    ["Every training product is worse, independently, than the model fitted to it. Labels "
     "are not a ceiling on accuracy.",
     "**RMSE, kappa and QWK** agree on the winner in every city, so the metric you pick "
     "doesn't change the answer.",
     "HCMC's zero-shot win is a coincidence of levels, not proof that transfer really "
     "works.",
     "Next step: a Vietnamese reference product with roads, and a deeper 2018 Sentinel-2 "
     "archive."],
)

# ===================================================================== thank you + contact
s = add(L_FINALE)
settxt(s, 0, "Thank you")

s = add(L_CONTACT)
settxt(s, 23, "Indirizzo / 02 2399 0000 / mail@polimi.it / www.polimi.it")

prs.save(OUT)
print("saved:", OUT)
print("slides:", len(prs.slides))
